In [ ]:
from pathlib import Path
import requests

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

FILES = {
    "appointments.csv": "https://coding-platform.s3.amazonaws.com/dev/lms/tickets/c1a9c6a2-5967-41fb-9133-cb6f78008699/qDCCmpyfss1kBJJB.csv",
    "clinics.csv": "https://coding-platform.s3.amazonaws.com/dev/lms/tickets/0fe7e34d-9470-47d8-b7b1-d54b77e009fe/jbFzPdd2gB2Ex2TY.csv",
    "followup_notes.csv": "https://coding-platform.s3.amazonaws.com/dev/lms/tickets/5e122459-7975-4534-957f-8397b0f9c03e/ecWvPsdV2FNmRZAa.csv",
}

for filename, url in FILES.items():
    file_path = DATA_DIR / filename
    if not file_path.exists():
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        file_path.write_bytes(response.content)
    print(f"{filename}: ready ({file_path.stat().st_size:,} bytes)")

appointments_path = DATA_DIR / "appointments.csv"
clinics_path = DATA_DIR / "clinics.csv"
notes_path = DATA_DIR / "followup_notes.csv"


appointments.csv: ready (84,008 bytes)
clinics.csv: ready (491 bytes)
followup_notes.csv: ready (2,336 bytes)


Task 1.1 — Read and validate with the csv module only

In [ ]:
import csv
import json

appointments_path = DATA_DIR / "appointments.csv"

with open(appointments_path, newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)
    headers = reader.fieldnames
    raw_rows = list(reader)

print("Header:", headers)
print("Rows read:", len(raw_rows))

def parse_row(row):
    converted = dict(row)
    converted["age"] = int(converted["age"])
    converted["fee_inr"] = int(converted["fee_inr"])
    converted["prior_no_shows"] = int(converted["prior_no_shows"])
    converted["distance_km"] = float(converted["distance_km"])
    return converted

accepted = []
rejected = []

for row in raw_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError as error:
        rejected.append(row)

with open("rejected_rows.json", "w", encoding="utf-8") as file:
    json.dump(rejected, file, indent=2)

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))
print("Rejected file saved: rejected_rows.json")

# Acceptance check from the brief
assert len(raw_rows) == 1020
assert len(accepted) == 988
assert len(rejected) == 32

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Rows read: 1020
Accepted: 988
Rejected: 32
Rejected file saved: rejected_rows.json


Task 1.2 — Inspect and clean, pass 1

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(appointments_path)

print("Initial shape:", df.shape)
print("\\nInfo:")
df.info()
print("\\nMissing values:")
print(df.isnull().sum())
print("\\nExact duplicate rows:", df.duplicated().sum())
print("\\nRaw booking_channel distinct values:", df["booking_channel"].nunique())

# (a) Drop exact duplicate rows
df = df.drop_duplicates()
print("After dropping exact duplicates:", len(df))

# (b) Standardise booking_channel
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Distinct booking_channel values after cleaning:",
      sorted(df["booking_channel"].dropna().unique()))

# (c) Convert dates in the specified way
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])

# (d) Drop rows with missing appt_date or status
df = df.dropna(subset=["appt_date", "status"])
print("After dropping missing appt_date/status:", len(df))

assert len(df) == 990

Initial shape: (1020, 14)
\nInfo:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
\nMissing values:
appointment_id      0
patient_id        

Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [ ]:
# (a) Invalid ages become missing
invalid_age_count = ((df["age"] < 1) | (df["age"] > 100)).sum()
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan
print("Invalid ages set to missing:", int(invalid_age_count))

# (b) Fill missing values using medians
age_median = df["age"].median()
distance_median = df["distance_km"].median()

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)

print("Age median used:", age_median)
print("Distance median used:", distance_median)

# (c) Calculate fences after filling gaps, then cap
def cap_iqr(series, column_name):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    capped_count = ((series < lower) | (series > upper)).sum()
    capped = pd.Series(np.clip(series.to_numpy(), lower, upper), index=series.index)
    print(f"{column_name}: Q1={q1}, Q3={q3}, IQR={iqr}, lower={lower}, upper={upper}")
    print(f"{column_name}: values capped={int(capped_count)}")
    return capped, int(capped_count), (q1, q3, lower, upper)

df["distance_km"], distance_capped, distance_fences = cap_iqr(df["distance_km"], "distance_km")
df["fee_inr"], fee_capped, fee_fences = cap_iqr(df["fee_inr"], "fee_inr")

# (d) Round to required precision
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("Largest fee after capping and rounding:", df["fee_inr"].max())

assert invalid_age_count == 5
assert distance_capped == 26
assert fee_capped == 3

Invalid ages set to missing: 5
Age median used: 48.0
Distance median used: 8.25
distance_km: Q1=4.625, Q3=13.0, IQR=8.375, lower=-7.9375, upper=25.5625
distance_km: values capped=26
fee_inr: Q1=500.0, Q3=787.5, IQR=287.5, lower=68.75, upper=1218.75
fee_inr: values capped=3
Largest fee after capping and rounding: 1219


Task 1.4 — Features and NumPy summary

In [ ]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

column_means = np.mean(X, axis=0)
print("Column means [age, distance_km, fee_inr, lead_days]:", column_means)

lead_mask = df["lead_days"].to_numpy() > 14
print("Rows with lead_days > 14:", int(lead_mask.sum()))
print("Share with lead_days > 14:", round(float(lead_mask.mean()) * 100, 1), "%")

distance_values = df["distance_km"].to_numpy()
distance_z = (distance_values - np.mean(distance_values)) / np.std(distance_values)
outlier_count = int((np.abs(distance_z) > 2).sum())
print("Distance rows with |z| > 2:", outlier_count)

print("Rows:", len(df))
print("No-shows:", int(df["no_show"].sum()))
print("No-show rate:", round(df["no_show"].mean() * 100, 1), "%")
print("Lead days min/max:", df["lead_days"].min(), df["lead_days"].max())

assert len(df) == 990
assert int(df["no_show"].sum()) == 218
assert int(lead_mask.sum()) == 296
assert outlier_count == 53

Column means [age, distance_km, fee_inr, lead_days]: [ 48.17373737   9.50090909 635.71414141  11.81919192]
Rows with lead_days > 14: 296
Share with lead_days > 14: 29.9 %
Distance rows with |z| > 2: 53
Rows: 990
No-shows: 218
No-show rate: 22.0 %
Lead days min/max: 1 45


Task 1.5 — Enrich clinics using the live Open-Meteo API

In [ ]:
clinics = pd.read_csv(clinics_path)

def get_clinic_temperature(latitude, longitude):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }
    last_reason = "Unknown API error"

    # Two attempts in total: the first request and one retry.
    for attempt in range(2):
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params=params,
                timeout=10
            )

            # Check the HTTP status before reading the JSON response.
            if response.status_code != 200:
                raise requests.exceptions.HTTPError(
                    f"HTTP status {response.status_code}"
                )

            payload = response.json()
            hourly = payload.get("hourly") if isinstance(payload, dict) else None
            temperatures = (
                hourly.get("temperature_2m")
                if isinstance(hourly, dict) else None
            )

            if not isinstance(temperatures, list) or len(temperatures) == 0:
                raise ValueError("hourly.temperature_2m is missing or empty")

            first_temperature = temperatures[0]
            if first_temperature is None:
                raise ValueError("first hourly temperature is missing")

            return float(first_temperature), None

        except requests.exceptions.RequestException as error:
            last_reason = f"Request failed: {error}"
        except (ValueError, TypeError, KeyError, AttributeError) as error:
            last_reason = f"Invalid API response: {error}"

    return None, last_reason

temperature_values = []
for _, clinic in clinics.iterrows():
    temp_c, reason = get_clinic_temperature(
        clinic["latitude"], clinic["longitude"]
    )
    temperature_values.append(temp_c)
    if reason is not None:
        print(f"{clinic['clinic_name']}: temperature unavailable after retry — {reason}")

clinics["temp_c"] = temperature_values
clinics.to_csv("clinics_enriched.csv", index=False)

print("Clinic rows saved:", len(clinics))
print(clinics[["clinic_name", "temp_c"]])


Clinic rows saved: 8
                clinic_name  temp_c
0     Kothrud Family Clinic    14.4
1      Hadapsar Care Centre    15.7
2    Sitabuldi Health Point    22.3
3      Malviya Nagar Clinic    15.3
4     Vaishali Nagar Clinic    15.3
5  Edappally Medical Centre    24.2
6        Vijay Nagar Clinic    15.2
7    Gomti Nagar Care Point    17.0


Task 1.6 — Export the clean table and SQL load script

In [ ]:
expected_columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show"
]

# Sort the clean data and format dates as YYYY-MM-DD text.
export_df = df.sort_values("appointment_id").copy()
export_df["booked_on"] = export_df["booked_on"].dt.strftime("%Y-%m-%d")
export_df["appt_date"] = export_df["appt_date"].dt.strftime("%Y-%m-%d")
export_df = export_df[expected_columns]

# Check uniqueness and missing values before exporting.
print("Unique appointment IDs:", export_df["appointment_id"].nunique())
print("Appointment ID is unique:", export_df["appointment_id"].is_unique)
print("Missing values in the 16 columns:")
print(export_df.isnull().sum())

assert export_df["appointment_id"].is_unique
assert export_df.isnull().sum().sum() == 0
assert export_df.shape == (990, 16)

export_df.to_csv("appointments_clean.csv", index=False)
print("Saved appointments_clean.csv", export_df.shape)

# SQL helpers: text is quoted, numbers are unquoted, missing values become NULL.
def sql_text(value):
    if pd.isna(value):
        return "NULL"
    return f"'{str(value)}'"

def sql_number(value):
    if pd.isna(value):
        return "NULL"
    return str(value)

with open("load_data.sql", "w", encoding="utf-8") as sql_file:
    # Insert clinics first because appointments reference clinics by foreign key.
    for _, row in clinics.iterrows():
        values = [
            sql_text(row["clinic_id"]),
            sql_text(row["clinic_name"]),
            sql_text(row["city"]),
            sql_text(row["tier"]),
            sql_number(row["latitude"]),
            sql_number(row["longitude"]),
            sql_number(row["temp_c"])
        ]
        sql_file.write(f"INSERT INTO clinics VALUES ({','.join(values)});\n")

    text_columns = {
        "appointment_id", "patient_id", "clinic_id", "booked_on",
        "appt_date", "gender", "booking_channel", "visit_type",
        "reminder_sent", "status"
    }

    # Insert appointment rows after all clinic rows.
    for _, row in export_df.iterrows():
        values = []
        for column in expected_columns:
            if column in text_columns:
                values.append(sql_text(row[column]))
            else:
                values.append(sql_number(row[column]))
        sql_file.write(f"INSERT INTO appointments VALUES ({','.join(values)});\n")

print("Total INSERT statements expected:", len(clinics) + len(export_df))
print("Saved load_data.sql")


Unique appointment IDs: 990
Appointment ID is unique: True
Missing values in the 16 columns:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64
Saved appointments_clean.csv (990, 16)
Total INSERT statements expected: 998
Saved load_data.sql


Part 3: Machine Learning (Tasks 3.1–3.5)

0. Import libraries and load the cleaned Part 1 data

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score, recall_score,
    f1_score, roc_auc_score
)
from IPython.display import display

# In Google Colab, upload appointments_clean.csv using the Files panel if needed.
df = pd.read_csv("appointments_clean.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing values:")
print(df.isnull().sum())

# Keep the appointment IDs separately for the final risk-score file.
appointment_ids = df["appointment_id"].copy()


FileNotFoundError: [Errno 2] No such file or directory: 'appointments_clean.csv'

3.1 Features, train/test split and baseline

In [ ]:
# Convert reminder_sent from Yes/No to 1/0.
reminder_binary = df["reminder_sent"].map({"Yes": 1, "No": 0})
if reminder_binary.isna().any():
    unexpected = df.loc[reminder_binary.isna(), "reminder_sent"].unique()
    raise ValueError(f"Unexpected reminder_sent values: {unexpected}")

# Build the feature dataframe in the specified order.
X = df[[
    "age", "distance_km", "fee_inr", "lead_days", "prior_no_shows"
]].copy()
X["reminder_sent"] = reminder_binary
X["booking_channel"] = df["booking_channel"]
X["visit_type"] = df["visit_type"]
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])

y = df["no_show"].astype(int)

print("Feature count:", X.shape[1])
print("Feature names:", X.columns.tolist())

# Keep original row order; do not sort or shuffle before this split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("No-shows in test set:", int(y_test.sum()))

# Baseline: predict Attended (0) for every test appointment.
baseline_predictions = np.zeros(len(y_test), dtype=int)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
print("Baseline accuracy (always Attended):", round(baseline_accuracy, 4))


3.2 Logistic regression

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

lr_pred = lr.predict(X_test_s)
lr_prob = lr.predict_proba(X_test_s)[:, 1]

print("Logistic regression accuracy:", round(accuracy_score(y_test, lr_pred), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, lr_pred))
print("Precision:", round(precision_score(y_test, lr_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, lr_pred, zero_division=0), 4))
print("F1:", round(f1_score(y_test, lr_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, lr_prob), 4))

coef_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lr.coef_[0]
})
print("Coefficients (scaled features):")
display(coef_table)

reminder_index = list(X.columns).index("reminder_sent")
reminder_coef = lr.coef_[0][reminder_index]
reminder_scale = scaler.scale_[reminder_index]

print("exp(scaled reminder coefficient):", np.exp(reminder_coef))
print("exp(coefficient / feature SD):", np.exp(reminder_coef / reminder_scale))

print(
    "Interpretation: In this model, holding the other features constant, "
    "the reminder Yes-versus-No association corresponds to an odds multiplier "
    f"of {np.exp(reminder_coef / reminder_scale):.3f}. This is an observational "
    "association and does not prove that reminders cause fewer no-shows."
)

3.3 Decision tree and overfitting

In [ ]:
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)
full_train_acc = accuracy_score(y_train, tree_full.predict(X_train))
full_test_acc = accuracy_score(y_test, tree_full.predict(X_test))

tree_depth3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_depth3.fit(X_train, y_train)
depth3_train_acc = accuracy_score(y_train, tree_depth3.predict(X_train))
depth3_test_acc = accuracy_score(y_test, tree_depth3.predict(X_test))

print("Full-depth tree training accuracy:", round(full_train_acc, 4))
print("Full-depth tree test accuracy:", round(full_test_acc, 4))
print("Full-depth tree depth:", tree_full.get_depth())
print("Depth-3 tree training accuracy:", round(depth3_train_acc, 4))
print("Depth-3 tree test accuracy:", round(depth3_test_acc, 4))

print(
    "Interpretation: A much higher training score than test score for the "
    "unrestricted tree indicates overfitting: it memorises patterns in the "
    "training data that do not generalise well to unseen bookings."
)

3.4 Random forest and test-set comparison table

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("Random forest training accuracy:", round(rf.score(X_train, y_train), 4))
print("Random forest test accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("Random forest recall:", round(recall_score(y_test, rf_pred, zero_division=0), 4))
print("Random forest ROC-AUC:", round(roc_auc_score(y_test, rf_prob), 4))

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print("Top three random-forest feature importances:")
display(importance.head(3))

comparison = pd.DataFrame([
    {
        "model": "Logistic regression",
        "accuracy": accuracy_score(y_test, lr_pred),
        "recall": recall_score(y_test, lr_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, lr_prob)
    },
    {
        "model": "Decision tree (depth 3)",
        "accuracy": accuracy_score(y_test, tree_depth3.predict(X_test)),
        "recall": recall_score(y_test, tree_depth3.predict(X_test), zero_division=0),
        "roc_auc": roc_auc_score(y_test, tree_depth3.predict_proba(X_test)[:, 1])
    },
    {
        "model": "Random forest (100 trees)",
        "accuracy": accuracy_score(y_test, rf_pred),
        "recall": recall_score(y_test, rf_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, rf_prob)
    }
])
print("Test-set comparison (reporting only):")
display(comparison.round(4))


3.5 Tune, choose the model and threshold using training data only

In [1]:
# (a) Tune random forest hyperparameters using training data only.
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 8]
    },
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)

print("Tuned forest best_params_:", rf_grid.best_params_)
print("Tuned forest best_score_ (5-fold CV ROC-AUC):", round(rf_grid.best_score_, 4))

# (b) Cross-validated ROC-AUC for all candidate models.
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

cv_models = {
    "Logistic regression pipeline": lr_pipe,
    "Decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random forest (100 trees)": RandomForestClassifier(n_estimators=100, random_state=42),
    "Tuned random forest": rf_grid.best_estimator_
}

cv_results = {}
for name, candidate in cv_models.items():
    if name == "Tuned random forest":
        # The grid search best_score_ is already the 5-fold training CV ROC-AUC.
        score = rf_grid.best_score_
    else:
        score = cross_val_score(
            candidate, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1
        ).mean()
    cv_results[name] = score

cv_table = pd.DataFrame({
    "model": list(cv_results.keys()),
    "training_cv_roc_auc": list(cv_results.values())
}).sort_values("training_cv_roc_auc", ascending=False)
print("Training-only model selection results:")
display(cv_table.round(4))

# Winner is selected only from training cross-validation scores.
winning_name = max(cv_results, key=cv_results.get)
winning_model = cv_models[winning_name]
print("Selected model:", winning_name)
print("Selected training CV ROC-AUC:", round(cv_results[winning_name], 4))


NameError: name 'GridSearchCV' is not defined

3.5(c) Choose the threshold using out-of-fold training probabilities

In [ ]:
oof_prob = cross_val_predict(
    winning_model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

threshold_rows = []
for threshold in [0.50, 0.35]:
    oof_pred = (oof_prob >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_train, oof_pred, zero_division=0),
        "recall": recall_score(y_train, oof_pred, zero_division=0),
        "f1": f1_score(y_train, oof_pred, zero_division=0),
        "number_flagged": int(oof_pred.sum())
    })

threshold_table = pd.DataFrame(threshold_rows)
print("Out-of-fold training threshold comparison:")
display(threshold_table.round(4))

# Select higher F1; if tied, select 0.35 because it offers the more recall-oriented threshold.
best_f1 = threshold_table["f1"].max()
best_rows = threshold_table[threshold_table["f1"] == best_f1]
if len(best_rows) > 1:
    chosen_threshold = 0.35
else:
    chosen_threshold = float(best_rows.iloc[0]["threshold"])

print("Chosen threshold:", chosen_threshold)
print(
    "Recall matters because a false negative means an at-risk appointment "
    "is not called and may leave a doctor slot empty; a false positive costs "
    "the call-centre team a short follow-up call."
)


3.5(d) Fit the selected model and evaluate once on the test set

In [ ]:
winning_model.fit(X_train, y_train)
test_prob = winning_model.predict_proba(X_test)[:, 1]
test_flag = (test_prob >= chosen_threshold).astype(int)

final_precision = precision_score(y_test, test_flag, zero_division=0)
final_recall = recall_score(y_test, test_flag, zero_division=0)
final_f1 = f1_score(y_test, test_flag, zero_division=0)
final_accuracy = accuracy_score(y_test, test_flag)
final_auc = roc_auc_score(y_test, test_prob)
number_flagged = int(test_flag.sum())

print("Final selected model:", winning_name)
print("Chosen threshold:", chosen_threshold)
print("Test precision:", round(final_precision, 4))
print("Test recall:", round(final_recall, 4))
print("Test F1:", round(final_f1, 4))
print("Test accuracy at chosen threshold:", round(final_accuracy, 4))
print("Test number flagged:", number_flagged)
print("Test ROC-AUC:", round(final_auc, 4))

print(
    "A clinic would set its operational threshold by balancing the number "
    "of follow-up calls the front desk can make each evening against the "
    "cost of leaving a doctor slot empty."
)

3.5(e) Save test-set risk scores

In [ ]:
# X_test.index preserves the original rows from the cleaned dataframe.
test_risk_scores = pd.DataFrame({
    "appointment_id": appointment_ids.loc[X_test.index].values,
    "no_show": y_test.values,
    "risk_prob": test_prob,
    "flag": test_flag
})

test_risk_scores.to_csv("test_risk_scores.csv", index=False)

print("Saved test_risk_scores.csv")
print("Output shape:", test_risk_scores.shape)
print("Flagged rows:", int(test_risk_scores["flag"].sum()))
display(test_risk_scores.head(10))
